# ir_calendar_inspect_record — 檢視一筆法說彙整與一份財報解析結果

- 用途：**人工檢視用**。挑一筆 `s_ir_calendar_summary`（法說結果爬蟲的 LLM 彙整），以及同公司同期別的一份財報檔案在 `s_ir_calendar_document_text`（全文）/ `s_ir_calendar_document_element`（元素）的內容：
  1. 先印**原始欄位**（每欄一行），確認資料長什麼樣
  2. 再印**整理後的 markdown 報告**，可直接複製給 user 看
- **只讀**：不寫表、不寫 Volume，可以隨時重跑。每次查詢都有篩選 + `limit`，不拉全表。
- 輸入：`s_{domain}_summary`、`s_{domain}_document_text`、`s_{domain}_document_element`（由 `consume_batches` / `parse_documents` 寫入，DDL 見 `init_tables` [c13] [c17] [c18]）
- 輸出：stdout
- 排程：不排程，人工執行
- 負責人 / 更新日期：（填）/ 2026-09-23

## 怎麼挑那一筆

| widget | 留空時 | 說明 |
|---|---|---|
| `company_key` | 最近更新的一筆 summary | `<市場>:<代號>`，如 `TPE:2353`、`HK:1070` |
| `period` | 該公司最新一期 | `yyyyQn`，如 `2026Q2` |
| `volume_path` | 同公司同期別、最近解析的一份文件 | 直接指定某份檔案（`s_document_text.volume_path`）；有填就不看 `company_key` / `period` 找文件 |
| `doc_kind` | 不限 | `Presentation` / `Financial_Statements` / `Press_Release` / `Earnings_Transcript` |
| `text_preview_chars` | `3000` | 全文 markdown 預覽字數 |
| `element_limit` | `30` | 印出前幾個元素 |
| `element_chars` | `300` | 每個元素內容截斷長度 |

[c10] / [c20] 會先列出候選清單（最多 10 / 20 筆），要換另一筆就把清單上的值填進 widget 再跑一次。

## cell

| cell | 內容 |
|---|---|
| [c03] `pure_helpers` | 純函式：截斷、原始欄位排版、兩份 markdown 報告（無 I/O，`tests/test_ir_calendar_inspect_record.py`） |
| [c10]～[c12] | summary：挑一筆 → 原始欄位 → 整理報告 |
| [c20]～[c23] | 文件：挑一份 → `document_text` 原始欄位 + 全文預覽 → `document_element` 統計與前 N 個元素 → 表格 / 圖表描述範例 |
| [c30] `report_user` | 合併兩份整理報告，印成一段 markdown 給 user |


In [ ]:
# [c01] params
# 只讀：這份 notebook 不寫表，可以隨時重跑。預設值取自 config/project.yml。
# 注意：widget 一旦建立過，改 code 的預設值不會更新既有 widget；要重設請先 dbutils.widgets.removeAll() 再跑本 cell。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")      # 表名中段；層級前綴 s_ 固定，見 docs/conventions.md 2.1
dbutils.widgets.text("company_key", "")            # 空 = 最近更新的一筆 summary
dbutils.widgets.text("period", "")                 # 空 = 該公司最新一期
dbutils.widgets.text("volume_path", "")            # 空 = 同公司同期別、最近解析的一份文件
dbutils.widgets.text("doc_kind", "")               # 空 = 不限
dbutils.widgets.text("text_preview_chars", "3000")  # 全文 markdown 預覽字數
dbutils.widgets.text("element_limit", "30")        # 印出前幾個元素
dbutils.widgets.text("element_chars", "300")       # 每個元素內容截斷長度

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "company_key": dbutils.widgets.get("company_key").strip(),
    "period": dbutils.widgets.get("period").strip(),
    "volume_path": dbutils.widgets.get("volume_path").strip(),
    "doc_kind": dbutils.widgets.get("doc_kind").strip(),
    "text_preview_chars": int(dbutils.widgets.get("text_preview_chars")),
    "element_limit": int(dbutils.widgets.get("element_limit")),
    "element_chars": int(dbutils.widgets.get("element_chars")),
}
assert cfg["catalog"] and cfg["schema"] and cfg["domain"], "catalog / schema / domain 不可為空"
assert cfg["element_limit"] >= 1, "element_limit 至少 1"
print(cfg)


In [ ]:
# [c02] imports
# 本 cell 與 [c03] 不碰 spark / dbutils，可被 tests/ 載入。
import json
from datetime import date, datetime

from pyspark.sql import functions as F

LAYER_PREFIX = {"bronze": "b_", "silver": "s_"}

# 報告裡「目錄」要列的元素種類；頁首頁尾頁碼在元素統計裡看得到，但不進報告內文。
# 值是 ai_parse_document 的 element type（實際出現哪些以 [c22] 統計為準）。
HEADER_TYPES = ("title", "section_header")
NOISE_TYPES = ("page_header", "page_footer", "page_number")


In [ ]:
# [c03] pure_helpers
# 純函式：輸入是 Row.asDict(recursive=True) 的 dict / list，輸出字串。無 I/O。


def clip(s, n: int) -> str:
    """None → ''；超過 n 字截斷並標出原長度。換行保留。"""
    if s is None:
        return ""
    s = str(s)
    return s if len(s) <= n else f"{s[:n]} …（共 {len(s):,} 字，已截斷）"


def one_line(s, n: int) -> str:
    """元素列表用：換行壓成 ⏎，再截斷。"""
    return clip(None if s is None else str(s).replace("\r", "").replace("\n", " ⏎ "), n)


def fmt_value(v, n: int = 500) -> str:
    """原始欄位值 → 一行字串。list / dict 轉 JSON（中文不跳脫），日期時間用 ISO。"""
    if v is None:
        return "NULL"
    if isinstance(v, (datetime, date)):
        return v.isoformat()
    if isinstance(v, (list, dict)):
        return clip(json.dumps(v, ensure_ascii=False, default=str), n)
    return clip(v, n)


def dump_row(title: str, d: dict, *, skip=(), n: int = 500) -> str:
    """一列資料直式排版：每欄一行 `欄位 = 值`，欄位名對齊。skip 的欄位不印（例如另外處理的長文字欄位）。"""
    keys = [k for k in d if k not in skip]
    w = max((len(k) for k in keys), default=0)
    lines = [f"===== {title} ====="]
    lines += [f"  {k:<{w}} = {fmt_value(d[k], n)}" for k in keys]
    return "\n".join(lines)


def pretty_json(s: str | None) -> str:
    """JSON 字串縮排；不是合法 JSON 就原樣回傳。"""
    if not s:
        return "NULL"
    try:
        return json.dumps(json.loads(s), ensure_ascii=False, indent=2)
    except (ValueError, TypeError):
        return s


def _bullets(items) -> list[str]:
    items = [str(x).strip() for x in (items or []) if x is not None and str(x).strip()]
    return [f"- {x}" for x in items] if items else ["- （無）"]


def _yn(v) -> str:
    return "—" if v is None else ("是" if v else "否")


def summary_report(s: dict) -> str:
    """s_ir_calendar_summary 一列 → 給 user 看的 markdown。"""
    head = f"{s.get('company_name') or ''}（{s.get('company_key') or ''}）{s.get('period') or ''} 法說彙整".strip()
    lines = [
        f"## {head}",
        "",
        "| 項目 | 內容 |",
        "|---|---|",
        f"| 公司 | {s.get('company_name') or '—'}（{s.get('stock_code') or '—'}，{s.get('market') or '—'}） |",
        f"| 分類 | {s.get('category') or '—'} |",
        f"| 期別 | {s.get('period') or '—'}（公司口徑：{s.get('fiscal_period') or '—'}） |",
        f"| 法說日期 | {fmt_value(s.get('conference_date')) if s.get('conference_date') else '—'}（台北日期） |",
        f"| 找到法說內容 | {_yn(s.get('found'))} |",
        f"| 以財報新聞稿替代 | {_yn(s.get('fallback'))} |",
        f"| 彙整時間 (UTC) | {fmt_value(s.get('crawled_at')) if s.get('crawled_at') else '—'} |",
        "",
        "### 重點",
        *_bullets(s.get("core_points")),
        "",
        "### 展望 / 指引",
        (s.get("guidance") or "（無）").strip(),
        "",
        "### 關鍵數字",
        *_bullets(s.get("key_numbers")),
        "",
        "### 風險",
        *_bullets(s.get("risks")),
    ]
    if s.get("notes"):
        lines += ["", "### 備註", str(s["notes"]).strip()]
    sources = s.get("sources") or []
    lines += ["", "### 來源"]
    if sources:
        for i, src in enumerate(sources, 1):
            src = src or {}
            media = f"（{src.get('media')}）" if src.get("media") else ""
            lines.append(f"{i}. {src.get('title') or '(無標題)'}{media} {src.get('url') or ''}".rstrip())
    else:
        lines.append("- （無）")
    return "\n".join(lines)


def element_line(e: dict, n: int) -> str:
    """元素一行：序號、頁、種類、內容（圖表沒有 content 時用 description）。"""
    body = e.get("content") or ""
    if not body.strip() and e.get("description"):
        body = "[圖表描述] " + e["description"]
    page = "—" if e.get("page_id") is None else e["page_id"] + 1
    return f"  #{e.get('seq')!s:>4}  p{page!s:<4} {e.get('element_type') or '?':<15} {one_line(body, n)}"


def document_report(doc: dict, type_counts: dict, headers: list[dict], samples: list[dict], *, n: int = 300) -> str:
    """s_document_text 一列 + element 統計 → 給 user 看的 markdown。
    type_counts：{element_type: 筆數}；headers：標題元素（依 seq）；samples：內文範例元素（依 seq）。"""
    size = doc.get("bytes")
    lines = [
        f"## 財報檔案解析：{doc.get('file_name') or doc.get('volume_path')}",
        "",
        "| 項目 | 內容 |",
        "|---|---|",
        f"| 公司 / 期別 | {doc.get('company_key') or '—'} / {doc.get('period') or '—'}（公司口徑：{doc.get('fiscal_label') or '—'}） |",
        f"| 文件種類 | {doc.get('doc_kind') or '—'} |",
        f"| 檔案大小 | {f'{size / 1024:,.0f} KB' if size else '—'} |",
        f"| 頁數 / 元素數 / 全文字數 | {doc.get('page_count') or 0} / {doc.get('element_count') or 0} / {doc.get('text_chars') or 0:,} |",
        f"| 解析狀態 | {doc.get('parse_status') or '—'}（{doc.get('parser') or ''} {doc.get('parser_version') or ''}） |",
        f"| 解析時間 (UTC) | {fmt_value(doc.get('parsed_at')) if doc.get('parsed_at') else '—'} |",
        f"| 檔案位置 | `{doc.get('volume_path')}` |",
        "",
        "### 內容組成",
    ]
    if type_counts:
        lines += [f"- {k}：{v}" for k, v in sorted(type_counts.items(), key=lambda kv: (-kv[1], kv[0]))]
    else:
        lines.append("- （element 表沒有這份文件的資料）")
    lines += ["", "### 章節標題"]
    if headers:
        for h in headers:
            page = "" if h.get("page_id") is None else f"（p{h['page_id'] + 1}）"
            lines.append(f"- {one_line(h.get('content'), 80)}{page}")
    else:
        lines.append("- （沒有 title / section_header 元素）")
    lines += ["", "### 內文節錄"]
    if samples:
        for e in samples:
            body = e.get("content") or ""
            if not body.strip() and e.get("description"):
                body = "[圖表描述] " + e["description"]
            page = "" if e.get("page_id") is None else f"p{e['page_id'] + 1} "
            lines.append(f"- {page}{e.get('element_type')}：{one_line(body, n)}")
    else:
        lines.append("- （無）")
    return "\n".join(lines)


def table_name(cfg: dict, layer: str, short: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{LAYER_PREFIX[layer]}{cfg['domain']}_{short}"


In [ ]:
# [c10] pick_summary
# 挑一筆 summary：有 company_key 就取該公司（period 空 = 最新一期），否則取最近更新的一筆。
# 先列候選清單（limit 10），要換別筆就把清單上的 company_key / period 填進 widget 再跑。
t_summary = table_name(cfg, "silver", "summary")
q = spark.table(t_summary)
if cfg["company_key"]:
    q = q.filter(F.col("company_key") == cfg["company_key"])
if cfg["period"]:
    q = q.filter(F.col("period") == cfg["period"])
order = ([F.col("period").desc(), F.col("updated_at").desc()] if cfg["company_key"]
         else [F.col("updated_at").desc(), F.col("period").desc()])

cand = (q.select("company_key", "company_name", "period", "conference_date", "found", "fallback", "updated_at")
        .orderBy(*order).limit(10).collect())
print(f"[{t_summary}] 候選（最多 10 筆，第一筆為本次檢視對象）")
for r in cand:
    print(f"  {r.company_key:<12} {r.period:<7} {r.conference_date!s:<10} found={r.found!s:<5} "
          f"fallback={r.fallback!s:<5} updated_at={r.updated_at}  {r.company_name or ''}")

summary = None
if cand:
    summary = (q.filter((F.col("company_key") == cand[0].company_key) & (F.col("period") == cand[0].period))
               .limit(1).collect()[0].asDict(recursive=True))
else:
    print(f"!!! 找不到符合的 summary（company_key={cfg['company_key']!r} period={cfg['period']!r}），後面文件改用 widget 條件找")


In [ ]:
# [c11] show_summary_raw
# 原始欄位：每欄一行；summary_json（LLM 產出的原文）另外縮排印，看有沒有 silver 沒展開的欄位。
if summary:
    print(dump_row(f"{t_summary}  {summary['company_key']} {summary['period']}", summary, skip=("summary_json",)))
    print("\n----- summary_json（原文，縮排）-----")
    print(pretty_json(summary.get("summary_json")))


In [ ]:
# [c12] report_summary
# 整理後給 user 看的版本（markdown）。
if summary:
    summary_md = summary_report(summary)
    print(summary_md)
else:
    summary_md = ""


In [ ]:
# [c20] pick_document
# 挑一份文件：volume_path 有填就用它；否則取 summary（或 widget）的 company_key + period；doc_kind 有填再篩。
# 找不到同公司同期別的文件時，改看全表最近解析的一份（並印出提醒）。
t_text = table_name(cfg, "silver", "document_text")
t_elem = table_name(cfg, "silver", "document_element")
doc_cols = ["volume_path", "sha256", "file_name", "company_key", "period", "doc_kind",
            "page_count", "element_count", "text_chars", "parse_status", "parsed_at"]


def _doc_candidates(df):
    if cfg["doc_kind"]:
        df = df.filter(F.col("doc_kind") == cfg["doc_kind"])
    return df.select(*doc_cols).orderBy(F.col("parsed_at").desc()).limit(20).collect()


df_text = spark.table(t_text)
if cfg["volume_path"]:
    docs = _doc_candidates(df_text.filter(F.col("volume_path") == cfg["volume_path"]))
else:
    ck = summary["company_key"] if summary else cfg["company_key"]
    pd_ = summary["period"] if summary else cfg["period"]
    d = df_text
    if ck:
        d = d.filter(F.col("company_key") == ck)
    if pd_:
        d = d.filter(F.col("period") == pd_)
    docs = _doc_candidates(d)
    if not docs and (ck or pd_):
        print(f"!!! {ck} {pd_} 沒有已解析的文件，改看全表最近解析的一份")
        docs = _doc_candidates(df_text)

print(f"[{t_text}] 候選（最多 20 筆，第一筆為本次檢視對象）")
for r in docs:
    print(f"  {r.doc_kind or '?':<20} pages={r.page_count!s:<4} elements={r.element_count!s:<5} "
          f"chars={r.text_chars!s:<8} {r.parse_status!s:<8} {r.parsed_at}  {r.volume_path}")
assert docs, f"{t_text} 沒有符合的文件：parse_documents 還沒跑過，或 volume_path / doc_kind 條件太嚴"

# 一份文件的鍵 = (volume_path, sha256)；sha256 可能是 NULL，用 eqNullSafe
key_cond = (F.col("volume_path") == docs[0].volume_path) & F.col("sha256").eqNullSafe(docs[0].sha256)
doc = df_text.filter(key_cond).limit(1).collect()[0].asDict(recursive=True)   # 單列，text_md 是整份全文


In [ ]:
# [c21] show_text_raw
# document_text 原始欄位 + 全文 markdown 預覽（前 text_preview_chars 字）。
print(dump_row(f"{t_text}  {doc['file_name']}", doc, skip=("text_md",)))
print(f"\n----- text_md 預覽（前 {cfg['text_preview_chars']:,} 字 / 全文 {len(doc.get('text_md') or ''):,} 字）-----")
print(clip(doc.get("text_md"), cfg["text_preview_chars"]) or "NULL")


In [ ]:
# [c22] show_elements
# document_element：先看元素種類 / 頁的分布，再印一個元素的完整欄位，最後逐行列前 element_limit 個元素。
# 只篩這一份文件（最多數千列），groupBy / limit 都在 cluster 上做。
df_el = spark.table(t_elem).filter(key_cond)

type_counts = {r.element_type or "(null)": r["count"] for r in df_el.groupBy("element_type").count().collect()}
agg = df_el.agg(F.count("*").alias("n"), F.countDistinct("page_id").alias("pages"),
                F.sum(F.when(F.col("description").isNotNull(), 1).otherwise(0)).alias("with_desc")).collect()[0]
print(f"[{t_elem}] 元素 {agg.n} 個，分布在 {agg.pages} 頁；有圖表描述的 {agg.with_desc} 個")
for k, v in sorted(type_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {k:<16} {v}")

elements = [r.asDict() for r in df_el.orderBy("seq").limit(cfg["element_limit"]).collect()]
if elements:
    print()
    print(dump_row("第一個元素（完整欄位）", elements[0], n=cfg["element_chars"]))
    print(f"\n----- 前 {len(elements)} 個元素（seq / 頁 / 種類 / 內容）-----")
    for e in elements:
        print(element_line(e, cfg["element_chars"]))
else:
    print("!!! element 表沒有這份文件：可能 parse_status 不是 success/partial，或 silver 還沒重算")


In [ ]:
# [c23] show_tables_figures
# 表格與圖表描述是最常要確認的：各印前 3 個的完整內容（表格通常是 HTML / markdown）。
for etype, field in (("table", "content"), ("figure", "description")):
    rows = (df_el.filter(F.col("element_type") == etype).orderBy("seq")
            .select("seq", "page_id", "content", "description").limit(3).collect())
    print(f"\n===== {etype}（前 {len(rows)} 個，看 {field}）=====")
    for r in rows:
        page = "—" if r.page_id is None else r.page_id + 1
        print(f"--- seq={r.seq} p{page}")
        print(clip(r[field] or r.content or r.description, 3000) or "NULL")


In [ ]:
# [c30] report_user
# 給 user 的整理版：summary + 文件概要（章節標題 + 前幾段內文）。整段 markdown 可直接複製貼上。
headers = [r.asDict() for r in df_el.filter(F.col("element_type").isin(list(HEADER_TYPES)))
           .orderBy("seq").select("seq", "page_id", "content").limit(30).collect()]
samples = [r.asDict() for r in df_el.filter(~F.col("element_type").isin(list(NOISE_TYPES + HEADER_TYPES)))
           .orderBy("seq").select("seq", "page_id", "element_type", "content", "description").limit(8).collect()]

report_md = "\n\n---\n\n".join(x for x in (
    summary_md,
    document_report(doc, type_counts, headers, samples, n=cfg["element_chars"]),
) if x)
print(report_md)
# 開發時想看排版後的樣子：displayHTML 需要 markdown 轉 HTML，這裡不另裝套件；貼到任何 markdown 檢視器即可。
